# Defining global variables

In [ ]:
REPO_NAME = 'Textual_Analysis_in_Finance'
BASE_DIR = f'/kaggle/working/{REPO_NAME}'
WEEK = 2

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Clone the lecture's git repo

In [ ]:
!git clone https://github.com/minhtriphan/{REPO_NAME}.git
%cd {REPO_NAME}

# Sentiment analysis with Financial Phrasebank dataset

## Financial Phrasebank dataset
#### Description

A polar sentiment dataset of sentences from Finnish financial news. The dataset consists of 4840 sentences from English language financial news categorised by sentiment. The sentiment labels are annotated by 16 annotators, in which 3 are researchers and 13 are Master students in finance, accounting, and economics at Aalto University. There are around 5000 sentences before entering the annotation process.

On average, each annotator is given 1500 sentences to annotate and they have 1 month to do the task. Each sentence is annotated by 5-8 annotators. The final dataset comprises on documents with the agreement >50% among the annotators. The dataset is split based on the agreement levels, which are 50%, 66%, 75%, and 100% levels.

**_This is one of a few public financial text datasets with sentiment labels_**

#### Other details
* Paper: [Good debt or bad debt: Detecting semantic orientations in economic texts](https://asistdl.onlinelibrary.wiley.com/doi/full/10.1002/asi.23062?casa_token=a5HKG1SkCrIAAAAA%3AP3-YNdbRNDDiW0PsExxMZDaDo2kgcQf77ZkU3MbMKcEVktRsVpTfJDk87iWcZV-9weIISKjB_evv4tDG)
* Data card: [HuggingFace dataset](https://huggingface.co/datasets/takala/financial_phrasebank)
* Where to find it? In our lecture's Git repo, `Data/FinancialPhrasebank`

## In our today coding session
We analyse the sentiment of the Financial Phrasebank dataset using the following techniques:

* Using the Harvard-IV dictionary
* Using the Loughran-McDonald sentiment dictionary
* A supervised learning with Bag-of-Words
* A supervised learning with TF-IDF

## Let's load the data to Python!

In [ ]:
import os
import pandas as pd

agreement_level = '50'    # There are '50', '66', '75', and 'All' agreement levels
financial_phrasebank = pd.read_csv(
    os.path.join(BASE_DIR, 'Data', 'FinancialPhrasebank', f'Sentences_{agreement_level}Agree.txt'), 
    sep = '@',
    encoding = 'latin-1',
    header = None,
    names = ['text', 'sentiment']
)
financial_phrasebank

## Sentiment analysis with casual sentiment dictionary

Firstly, we have to **clean the texts**. Let's reuse the code from Week 1

In [ ]:
from Week_1.Code.Utils import readfile, lowercase, special_character_removal, fix_contraction, stopword_removal, lemmatization

def text_normalization(text):
    text = lowercase(text)
    text = special_character_removal(text, keep = None, remove_number = True)
    text = fix_contraction(text)
    text = stopword_removal(text, remove_single_letters = True)
    text = lemmatization(text)
    return text

financial_phrasebank['clean_text'] = financial_phrasebank['text'].apply(text_normalization)

* If you want to monitor the process, do this

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

financial_phrasebank['clean_text'] = financial_phrasebank['text'].progress_apply(text_normalization)    # Instead of .apply, use .progress_apply
financial_phrasebank.head()

#### Using the Harvard-IV dictionary

The easiest way to do the dictionary-based sentiment analysis is to use the `pysentiment2` [package](https://nickderobertis.github.io/pysentiment/index.html).

The outcome of this package includes:
* `Positive`: the number of positive words in the sentences based on the chosen dictionary
* `Negative`: the number of negative words in the sentences based on the chosen dictionary
* `Polarity`, where $Polarity = \frac{Positive - Negative}{Positive + Negative}$
* `Subjectivity`, where $Subjectivity = \frac{Positive + Negative}{Document length}$

In [ ]:
!pip install pysentiment2

* Use this package for a sample document

In [ ]:
text = financial_phrasebank['text'].iloc[2]
clean_text = financial_phrasebank['clean_text'].iloc[2]

print(text, '\n', '*' * 10, '\n', clean_text)

In [ ]:
import pysentiment2 as ps
hiv4 = ps.HIV4()
tokens = hiv4.tokenize(clean_text)
score = hiv4.get_score(tokens)
score

#### <span style="color:red">Your turn:</span> Can you write code to analyze the sentiment for all sentences in the dataset using the Harvard-IV dictionary?

*Hints:* Write a function that takes the `clean_text` (`str`) as the only argument, then output a `pd.Series` as follows:

```
pd.Series(
    data = Positive, Negative, Polarity, Subjectivity,
    index = ['pos_hiv', 'neg_hiv', 'polarity_hiv', 'subjectivity_hiv']
)
```

where `Positive`, `Negative`, `Polarity`, and `Subjectivity` are the output from the dictionary as above

**_The result should look like below_**

In [ ]:
'''YOUR CODE HERE'''

In [ ]:
financial_phrasebank[['pos_hiv', 'neg_hiv', 'polarity_hiv', 'subjectivity_hiv']]

#### Using the Loughran-McDonald (LM) dictionary

This is exactly the same as the usage of the Harvard-IV we have just done, just replace `ps.HIV` by `ps.LM`

In [ ]:
lm = ps.LM()
tokens = lm.tokenize(clean_text)
score = lm.get_score(tokens)
score

#### <span style="color:red">Your turn:</span> Can you write code to analyze the sentiment for all sentences in the dataset using the LM dictionary?

In [ ]:
'''YOUR CODE HERE'''

In [ ]:
financial_phrasebank[['pos_lm', 'neg_lm', 'polarity_lm', 'subjectivity_lm']]

#### Convert sentiment scores (`Polarity`) into sentiment prediction

We need a rule to convert polarity scores to sentiment predictions. A common rule is:

```
if polarity > 0:
    sentiment = 'positive'
elif polarity == 0:
    sentiment = 'neutral'
else:
    sentiment = 'negative'
```

In [ ]:
def convert_polarity_to_sentiment_prediction(polarity):
    if polarity > 0:
        sentiment = 'positive'
    elif polarity == 0:
        sentiment = 'neutral'
    else:
        sentiment = 'negative'
    return sentiment

financial_phrasebank['sentiment_pred_hiv'] = financial_phrasebank['polarity_hiv'].apply(convert_polarity_to_sentiment_prediction)
financial_phrasebank['sentiment_pred_lm'] = financial_phrasebank['polarity_lm'].apply(convert_polarity_to_sentiment_prediction)

## Model evaluation

Before moving to the supervised sentiment analysis, let's learn how to evaluate the sentiment predictions.

#### Choosing the evaluation metrics

This is a multi-class classification, where there are 3 classes, positive, negative, and neutral. A popular metric is F1 score.

$F_1 = 2\frac{Precision \times Recall}{Precision + Recall}$

where $Precision = \frac{TP}{TP + FP}$ and $Recall = \frac{TP}{TP + FN}$;

* $TP$: True positive - predict a yes when the true is yes
* $FP$: False positive - predict a yes when the true is no (also called, **Type I error**)
* $FN$: False negative - predict a no when the true is yes (also called, **Type II error**)

F1 is prefered than Accuracy because it takes into account the **imbalancedness** of the label

In [ ]:
from sklearn.metrics import f1_score

score_hiv = f1_score(financial_phrasebank['sentiment'], financial_phrasebank['sentiment_pred_hiv'], average = 'micro')
score_lm = f1_score(financial_phrasebank['sentiment'], financial_phrasebank['sentiment_pred_lm'], average = 'micro')
print(f'F1 scores of Harvard-IV and LM dictionaries are: {score_hiv} and {score_lm}, respectively')

We can also show the **confusion matrix** of each method

A confusion matrix compares actual ground truth labels with predicted classifications. Rows show the true values and columns show the predictations

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(financial_phrasebank['sentiment'], financial_phrasebank['sentiment_pred_hiv'], labels = ['negative', 'neutral', 'positive'])

The first (0, 0) entry, 146, means there are 146 sentences labeled as `negative` and the Harvard-IV dictionary predicts correctly as `negative`. The (1, 0) entry, 207, means there are 207 sentences with `neutral` labels but are predicted as `negative`. The (0, 1) entry, 171, means there are 171 `negative` sentences but predicted as `neutral` by the Harvard-IV dictionary.

In [ ]:
confusion_matrix(financial_phrasebank['sentiment'], financial_phrasebank['sentiment_pred_lm'], labels = ['negative', 'neutral', 'positive'])

#### <span style="color:red">Question</span>

1. What do you think about Harvard-IV predictions? How do LM predictions look like?

## Supervised learning sentiment analysis

In general, a supervised learning model is:

$y = f(X) + e$

where $y$ is the target, which is the sentiment label in our case; and $X$ is the set of features, which are extracted from our text. In this lecture, we use the Bag-of-Words and TF-IDF to extract $X$.

Firstly, the label column, `sentiment`, is currently a string column, we need to encode it to numerical column

In [ ]:
LABEL_ENCODE = {
    'neutral': 0,
    'negative': 1,
    'positive': 2
}
LABEL_ENCODE_INV = {v: k for k, v in LABEL_ENCODE.items()}

financial_phrasebank['sentiment_encoded'] = financial_phrasebank['sentiment'].map(LABEL_ENCODE)

**Important:** to prevent the **overfitting** problem, we have to split the dataset into 2 sets, one for training and for testing. ***Disclaimer: Because we will use only multi-class logistic regression, there is no hyperparameter, we don't need a validation set. In a normal ML setup, we have to split the data into 3 sets, for training, validation, and testing.***

In [ ]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    financial_phrasebank, 
    test_size = 0.5,
    random_state = 1,
    stratify = financial_phrasebank['sentiment_encoded']
)

#### <span style="color:red">Question</span>

2. Why do we need this line: `stratify = financial_phrasebank['sentiment_encoded']`?

## Extract features using Bag-of-Words

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_data['clean_text'])
y_train = train_data['sentiment_encoded']

################## IMPORTANT: use .transform here, not fit or fit_transform ##################
X_test = vectorizer.transform(test_data['clean_text'])
y_test = test_data['sentiment_encoded']

#### Build the model

We use the `sklearn.linear_model.LogisticRegression` object: [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

Important arguments:

* `penalty`: if we want to use normal logistic regression, set this argument to `None`
* `C`: the regularization hyperparameter
* `l1_ratio`: the Lasso-regularization ratio
* `random_state`: control for the random state

In [ ]:
from sklearn.linear_model import LogisticRegression

# Construct the model
model = LogisticRegression(penalty = None)
model.fit(X_train, y_train)

# Predict
test_data['sentiment_pred_bow_logit'] = model.predict(X_test)
test_data['sentiment_pred_bow_logit'] = test_data['sentiment_pred_bow_logit'].map(LABEL_ENCODE_INV)
test_data

In [ ]:
score_hiv = f1_score(test_data['sentiment'], test_data['sentiment_pred_hiv'], average = 'micro')
score_lm = f1_score(test_data['sentiment'], test_data['sentiment_pred_lm'], average = 'micro')
score_bow_logit = f1_score(test_data['sentiment'], test_data['sentiment_pred_bow_logit'], average = 'micro')

print('F1 scores:')
print('- Harvard-IV dictionary:', score_hiv)
print('- LM dictionary:', score_lm)
print('- Bag-of-Words + Logit:', score_bow_logit)

## Extract features using TF-IDF

#### <span style="color:red">Your turn:</span> implement a sentiment analysis using TF-IDF

**Important!** Use the `train_data` and `test_data` as above, please don't change them!

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

'''YOUR CODE TO EXTRACT TF-IDF FEATURES FROM TRAIN_DATA AND TEST_DATA'''

'''YOUR CODE FOR MODELING'''

'''YOUR CODE TO POST-PROCESS THE PREDICTIONS'''

## Final results

In [ ]:
score_hiv = f1_score(test_data['sentiment'], test_data['sentiment_pred_hiv'], average = 'micro')
score_lm = f1_score(test_data['sentiment'], test_data['sentiment_pred_lm'], average = 'micro')
score_bow_logit = f1_score(test_data['sentiment'], test_data['sentiment_pred_bow_logit'], average = 'micro')
score_tfidf_logit = f1_score(test_data['sentiment'], test_data['sentiment_pred_tfidf_logit'], average = 'micro')

print('F1 scores:')
print('- Harvard-IV dictionary:', score_hiv)
print('- LM dictionary:', score_lm)
print('- Bag-of-Words + Logit:', score_bow_logit)
print('- TF-IDF + Logit:', score_tfidf_logit)

#### <span style="color:red">Question</span>

3. What do you think about the results?

# Topic modeling with Latent Dirichlet Allocation (LDA)

In this section, we will apply LDA to analyze topics usually mentioned by central bankers in their communications

The data we will use is the central bankers' speeches, published in BIS website. To download the data, refer to the instruction [**here**](https://github.com/minhtriphan/Textual_Analysis_in_Finance/tree/main/Data/CentralBankerSpeeches).

In [ ]:
import gdown

url = 'https://drive.google.com/file/d/1CnadkUgrHYDwxepPhZMoD1UbR4XoTJnm/view?usp=sharing'
file_id = url.split('/')[-2]
download_url = f'https://drive.google.com/uc?id={file_id}'
output = 'central_banker_speeches.csv'
gdown.download(download_url, output, quiet = False)

central_banker_speeches = pd.read_csv('central_banker_speeches.csv')
central_banker_speeches = central_banker_speeches[['title', 'date', 'text']]
central_banker_speeches

The dataset is too large, let's focus on speeches in 2025

In [ ]:
central_banker_speeches['date'] = pd.to_datetime(central_banker_speeches['date'], format = '%Y-%m-%d %H:%M:%S')
central_banker_speeches = central_banker_speeches[central_banker_speeches['date'].dt.year == 2025]
central_banker_speeches

#### Normalize the texts

Again, we normalize the texts using our code in Week 1

In [ ]:
'''YOUR CODE HERE'''
central_banker_speeches['clean_text'] = ...

#### Let's start doing LDA using a package called `gensim`

The steps include:

1. Generate vocabulary
2. Convert the corpus to Bag-of-Words
3. Train the LDA model
4. Print the result

#### Step 1: Generate vocabulary (or dictionary)

This step is done using the method `corpora.Dictionary`. It takes a list of list of tokens (words) as the input. The input looks like:

```
texts = [
    ['word_1,1', 'word_1,2', ..., 'word_1,T_1'],
    ['word_2,1', 'word_2,2', ..., 'word_2,T_2'],
    ...
    ['word_N,1', 'word_N,2', ..., 'word_N,T_N'],
]
```

In [ ]:
from gensim.models import LdaModel
from gensim import corpora

# Generate the input
'''YOUR CODE HERE'''
texts = ...

# Generate the dictionary
dictionary = corpora.Dictionary(texts)
dictionary

#### Step 2: Convert the corpus to Bag-of-Words

This step is done using the method `dictionary.doc2bow` which takes the list of all words in a document as the input. Sample usage is:

```
document_1 = ['word_1,1', 'word_1,2', ..., 'word_1,T_1']
dictionary.doc2bow(document_1)
```

In [ ]:
corpus = [dictionary.doc2bow(text) for text in texts]

#### Step 3: Train the LDA model

This step is done using the `LdaModel` method, the usage is in the cell below. Important arguments include:

* `corpus`: The Bag-of-Words corpus we constructed in Step 2
* `id2word`: The dictionary we constructed in Step 1
* `num_topics`: The number of topics you want to extract (the most important/interesting argument)
* `random_state`: Control for randomness
* `passes`: The number of training iterations

In [ ]:
lda_model = LdaModel(
    corpus = corpus,
    id2word = dictionary,
    num_topics = 10,
    random_state = 42,
    passes = 10
)

#### Inspecting the results

First, let's check the topic words

In [ ]:
topics = lda_model.print_topics(num_words = 10)
for topic in topics:
    print(topic)

#### <span style="color:red">Question</span>

3. What are those numbers in front of words mean?

Infer topic distribution of a document

In [ ]:
doc = central_banker_speeches['clean_text'].iloc[0].split()
bow = dictionary.doc2bow(doc)

print(lda_model.get_document_topics(bow))

#### <span style="color:red">Question</span>

4. What do those numbers mean?